# 06 GPS-Simulation

Der Datensatz enthält keine Koordinaten. Da Maßnahmen ortsbezogen geplant werden, werden GPS-Koordinaten simuliert (in Absprache mit dem Dozenten):
- Je Unfall eine Koordinate (nicht je Fahrzeug).
- Jeder Gebietstyp erhält feste Ankerpunkte im Stadtgebiet; die Koordinate streut zufällig um einen Anker (ca. 1 km).
- Unfälle mit Gebiet 'Other/Unknown' werden gleichmaessig ueber das Stadtgebiet verteilt.

In [1]:
import pandas as pd
import numpy as np

df_clean  = pd.read_parquet('../../data/df_features.parquet')
df_unfall = pd.read_parquet('../../data/df_unfall.parquet')
print('df_unfall:', len(df_unfall), '| df_clean:', len(df_clean))

df_unfall: 9517 | df_clean: 12316


In [2]:
rng = np.random.default_rng(42)
BBOX = dict(lat=(8.86, 9.08), lon=(38.68, 38.88))
anker = {
    'Office areas':        [(9.012, 38.765), (9.010, 38.745), (8.997, 38.787)],
    'Residential areas':   [(9.022, 38.840), (9.030, 38.880), (9.030, 38.700),
                            (8.990, 38.800), (8.960, 38.750)],
    'Market areas':        [(9.035, 38.738)],
    'Church areas':        [(9.032, 38.752), (8.985, 38.770)],
    'Industrial areas':    [(8.950, 38.780), (8.930, 38.760)],
    'School areas':        [(9.020, 38.770), (8.980, 38.740)],
    'Recreational areas':  [(9.000, 38.760)],
    'Hospital areas':      [(9.015, 38.755)],
    'Outside rural areas': [(8.880, 38.700), (9.070, 38.860)],
}

lat, lon = [], []
for gebiet in df_unfall['Area_accident_occured']:
    if gebiet in anker:
        a = anker[gebiet][rng.integers(len(anker[gebiet]))]
        lat.append(a[0] + rng.normal(0, 0.008))
        lon.append(a[1] + rng.normal(0, 0.008))
    else:
        lat.append(rng.uniform(*BBOX['lat']))
        lon.append(rng.uniform(*BBOX['lon']))

df_unfall['Lat'] = np.clip(lat, *BBOX['lat'])
df_unfall['Lon'] = np.clip(lon, *BBOX['lon'])
df_unfall['GPS_simuliert'] = 'ja'

df_clean = df_clean.drop(columns=['Lat', 'Lon'], errors='ignore').merge(
    df_unfall[['Unfall_ID', 'Lat', 'Lon']], on='Unfall_ID', how='left')

df_unfall[['Unfall_ID', 'Area_accident_occured', 'Lat', 'Lon']].head()

,Unfall_ID,Area_accident_occured,Lat,Lon
0,0,Residential areas,9.013680,38.846004
1,1,Office areas,9.004525,38.771392
2,2,Recreational areas,8.989583,38.761023
3,3,Office areas,8.996866,38.780176
4,4,Industrial areas,8.937035,38.766222


## Speichern

In [3]:
df_unfall.to_parquet('../../data/df_unfall_gps.parquet', index=False)
df_clean.to_parquet('../../data/df_features_gps.parquet', index=False)
print('Gespeichert: df_unfall_gps.parquet', df_unfall.shape)
print('Gespeichert: df_features_gps.parquet', df_clean.shape)

Gespeichert: df_unfall_gps.parquet (9517, 44)
Gespeichert: df_features_gps.parquet (12316, 43)
